In [0]:
from pyspark.sql.functions import col

# Load Silver table
df = spark.table("crypto_ml.cryptosilver.btc_features")

# Show schema
df.printSchema()

# Show first rows
display(df)

# Count total rows
print("Total rows before cleaning:", df.count())

In [0]:
# Use correct column names from your table

selected_cols = ["date", "close", "rsi14", "volume", "return_1d"]

df_clean = df.select(*selected_cols)

# Remove nulls
df_clean = df_clean.dropna()

# Sort by date
df_clean = df_clean.orderBy("date")

print("Total rows after cleaning:", df_clean.count())
display(df_clean)

In [0]:
# Save cleaned data as temp view for next steps
df_clean.createOrReplaceTempView("btc_ml_clean")

print("Clean ML data is ready")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

# Use cleaned data
ml_df = df_clean.select("date", "close", "rsi14", "volume", "return_1d")

# Rename target column to label for Spark ML
ml_df = ml_df.withColumnRenamed("close", "label")

# Create features vector
assembler = VectorAssembler(
    inputCols=["rsi14", "volume", "return_1d"],
    outputCol="features"
)

model_df = assembler.transform(ml_df).select("date", "features", "label")

display(model_df)
print("Prepared model dataset count:", model_df.count())

In [0]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction"
)

lr_model = lr.fit(train_df)

print("Linear Regression model trained successfully")

In [0]:
lr_predictions = lr_model.transform(test_df)

display(
    lr_predictions.select("date", "label", "prediction").orderBy(col("date").desc())
)

In [0]:
lr_predictions.createOrReplaceTempView("btc_lr_predictions_temp")
print("Linear Regression predictions ready")

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE
rmse_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)
rmse = rmse_evaluator.evaluate(lr_predictions)

# MAE
mae_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mae"
)
mae = mae_evaluator.evaluate(lr_predictions)

# R2
r2_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)
r2 = r2_evaluator.evaluate(lr_predictions)

print("Linear Regression Evaluation Metrics")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

In [0]:
from pyspark.sql.functions import round, current_timestamp

lr_gold = lr_predictions.select(
    col("date"),
    round(col("label"), 2).alias("actual_close"),
    round(col("prediction"), 2).alias("predicted_close")
).orderBy(col("date").desc())

display(lr_gold)

In [0]:
from pyspark.sql.functions import lit

lr_gold = lr_gold.withColumn("model_name", lit("Linear Regression")) \
                 .withColumn("saved_at", current_timestamp())

display(lr_gold)

In [0]:
lr_gold.write.mode("overwrite").saveAsTable("crypto_ml.cryptogold.btc_lr_predictions")
print("Saved successfully to crypto_ml.cryptogold.btc_lr_predictions")

In [0]:
display(
    spark.table("crypto_ml.cryptogold.btc_lr_predictions").orderBy(col("date").desc())
)

In [0]:
# Select only date and close for ARIMA
arima_spark_df = df_clean.select("date", "close").orderBy("date")

display(arima_spark_df)
print("ARIMA input row count:", arima_spark_df.count())

In [0]:
import pandas as pd

arima_pdf = arima_spark_df.toPandas()

# Convert date column properly
arima_pdf["date"] = pd.to_datetime(arima_pdf["date"])

# Sort and set date index
arima_pdf = arima_pdf.sort_values("date")
arima_pdf = arima_pdf.set_index("date")

print(arima_pdf.tail())

In [0]:
%pip install statsmodels

In [0]:


from statsmodels.tsa.arima.model import ARIMA

# ARIMA on close prices
arima_model = ARIMA(arima_pdf["close"], order=(5, 1, 0))
arima_model_fit = arima_model.fit()

print("ARIMA model trained successfully")

In [0]:
forecast_steps = 7

forecast_values = arima_model_fit.forecast(steps=forecast_steps)

print("ARIMA Forecast:")
print(forecast_values)

In [0]:
last_date = arima_pdf.index.max()

future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=forecast_steps,
    freq="D"
)

forecast_df = pd.DataFrame({
    "date": future_dates,
    "predicted_close": forecast_values.values
})

print(forecast_df)

In [0]:
arima_forecast_spark = spark.createDataFrame(forecast_df)

display(arima_forecast_spark.orderBy(col("date").desc()))

In [0]:
from pyspark.sql.functions import lit, current_timestamp, round, col

arima_gold = arima_forecast_spark.select(
    col("date"),
    round(col("predicted_close"), 2).alias("predicted_close")
).withColumn("model_name", lit("ARIMA")) \
 .withColumn("saved_at", current_timestamp())

display(arima_gold.orderBy(col("date").desc()))

In [0]:
arima_gold.write.mode("overwrite").saveAsTable("crypto_ml.cryptogold.btc_arima_predictions")
print("Saved successfully to crypto_ml.cryptogold.btc_arima_predictions")

In [0]:
display(
    spark.table("crypto_ml.cryptogold.btc_arima_predictions").orderBy(col("date").desc())
)